In [ ]:
import math
import csv
import random
from google.colab import files

# ================= MPC / ACC CONSTANTS (From mpc_ml.py) =================
VSET = 0.38        # Target cruise speed (m/s)
HW = 0.70          # Headway time (seconds)
SD = 0.25          # Standstill distance (m)
TS = 0.20          # Sample time (seconds)
MIN_PWM = 95       # Motor start threshold
MAX_PWM = 255      # Motor max

# ================= SIMULATION CONFIG =================
TARGET_ROWS = 8000
CSV_FILENAME = "mpc_acc_simulated_8000.csv"
CSV_HEADERS = [
    "time", "dist", "d_safe", "df", "speed",
    "v_leader", "v_ref", "uff", "u_mpc", "u_total", "pwm", "pred_penalty"
]

data_log = []
current_time = 0.0

# Initial Robot State
dist = 1.5
speed = 0.0

print(f"Generating {TARGET_ROWS} rows of MPC telemetry data...")

for i in range(TARGET_ROWS):
    # 1. Simulate Leader Vehicle Environment
    # The leader vehicle smoothly speeds up, slows down, and occasionally stops
    v_leader = 0.2 + 0.18 * math.sin(current_time / 8.0)
    if v_leader < 0.05:
        v_leader = 0.0

    # 2. Update physical distance based on relative speeds
    dist += (v_leader - speed) * TS

    # Add minor sensor noise to make the dataset realistic for machine learning
    measured_dist = dist + random.uniform(-0.01, 0.01)
    measured_speed = speed + random.uniform(-0.005, 0.005)
    measured_speed = max(0.0, measured_speed)

    # 3. ACC Core Logic (Approximating your MPC Constraints)
    d_safe = SD + HW * measured_speed
    df = measured_dist - d_safe  # Distance Error

    # Calculate Reference Velocity (v_ref)
    if measured_dist > 2.0:
        v_ref = VSET # Open road, cruise at max speed
    else:
        # Match leader speed but correct for distance error
        v_ref = v_leader + 0.8 * df
        v_ref = max(0.0, min(VSET, v_ref))

    # Emergency Hard Stop
    if measured_dist < SD * 0.8:
        v_ref = 0.0

    # 4. Controller Output Generation
    # Feedforward (uff): Base power required to maintain v_ref
    uff = v_ref / 0.5  # Normalize against estimated max physical speed

    # Feedback (u_mpc): MPC correction for velocity and distance errors
    v_error = v_ref - measured_speed
    u_mpc = 0.5 * v_error + 0.1 * df

    # Total Control Effort
    u_total = uff + u_mpc
    u_total = max(0.0, min(1.0, u_total)) # Clamp between 0.0 and 1.0

    # Convert to PWM for the L298N Motor Driver
    if u_total > 0.01:
        # Map u_total to the viable PWM range
        pwm = int(MIN_PWM + u_total * (MAX_PWM - MIN_PWM))
        pwm = min(MAX_PWM, pwm)
    else:
        pwm = 0

    # Cut power entirely if closer than standstill distance
    if measured_dist <= SD:
        pwm = 0
        u_total = 0.0

    # 5. Simulate Robot Physics
    # The robot's actual speed responds to the PWM with a slight momentum delay
    target_sim_speed = (pwm / 255.0) * 0.45
    speed += (target_sim_speed - speed) * 0.15
    speed = max(0.0, speed)

    # 6. Log exact columns expected by mpc_ml_mlp.ipynb
    data_log.append([
        round(current_time, 3),
        round(measured_dist, 4),
        round(d_safe, 4),
        round(df, 4),
        round(measured_speed, 4),
        round(v_leader, 4),
        round(v_ref, 4),
        round(uff, 4),
        round(u_mpc, 4),
        round(u_total, 4),
        pwm,
        0.0000 # pred_penalty (simulated flat)
    ])

    current_time += TS

# ================= SAVE AND DOWNLOAD =================
print(f"Saving data to '{CSV_FILENAME}'...")
with open(CSV_FILENAME, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(CSV_HEADERS)
    writer.writerows(data_log)

print("Generation complete! Triggering download...")
files.download(CSV_FILENAME)

Generating 8000 rows of MPC telemetry data...
Saving data to 'mpc_acc_simulated_8000.csv'...
Generation complete! Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>